# 📚 CSIC Metadata Snapshot Exploration

This notebook explores the local parquet snapshot stored in `data/csic_metadata_snapshot/data/*.parquet` using **DuckDB** and **Pandas**.



In [ ]:
import duckdb
import pandas as pd

# Adjust pandas display limits for readability
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', None)

DATA_PATH = "../data/csic_metadata_snapshot/data/*.parquet"

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.5


## 1. Schema & Total Count

Check the schema of the parquet files and the total number of paper records.

In [4]:
# Inspect schema and column types
schema_df = con.sql(f"DESCRIBE SELECT * FROM '{DATA_PATH}'").df()
display(schema_df)

# Total count of papers
total_count = con.sql(f"SELECT count(*) AS total_papers FROM '{DATA_PATH}'").fetchone()[0]
print(f"\nTotal Papers in Dataset: {total_count:,}")

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,handle_uri,VARCHAR,YES,None,None,None
2,title,VARCHAR,YES,None,None,None
3,title_alternative,VARCHAR[],YES,None,None,None
4,authors,VARCHAR[],YES,None,None,None
5,author_cris_ids,VARCHAR[],YES,None,None,None
6,editors,VARCHAR[],YES,None,None,None
7,orcids,VARCHAR[],YES,None,None,None
8,abstract,VARCHAR,YES,None,None,None
9,description,VARCHAR,YES,None,None,None



Total Papers in Dataset: 415,130


## 2. Duplicate ID Check

Check if there are any duplicate records sharing the same CSIC `id`.


In [5]:
# Check for duplicate arXiv IDs
dup_check_df = con.sql(f"""
    SELECT
        count(*) AS total_records,
        count(DISTINCT id) AS distinct_ids,
        count(*) - count(DISTINCT id) AS duplicate_count
    FROM '{DATA_PATH}'
""").df()

display(dup_check_df)

dup_count = dup_check_df["duplicate_count"].iloc[0]
if dup_count > 0:
    print(f"\n⚠️ Found {dup_count:,} duplicate arXiv ID records.")
    sample_dups = con.sql(f"""
        SELECT
            id,
            count(*) AS occurrences
        FROM '{DATA_PATH}'
        GROUP BY id
        HAVING count(*) > 1
        ORDER BY occurrences DESC
        LIMIT 10
    """).df()
    print("Sample duplicate IDs:")
    display(sample_dups)
else:
    print("\n✅ No duplicate arXiv IDs found. All records are unique.")


,total_records,distinct_ids,duplicate_count
0,415130,415130,0



✅ No duplicate arXiv IDs found. All records are unique.
